# Task-v2.4 SpAM Simulation & MDS Evaluation Pipeline

This notebook mirrors `evaluation_task_v2_3.ipynb`, but simulates subjects under the
**task-v2.4** design: the task-v2.3 per-subject trial allocation **plus the
`frac_trials_repeated` whole-trial-repeat lever**. Concretely, on top of everything v2.3 does,
each subject now has `n_repeats = round(frac_trials_repeated * trials_per_subject)` of their
trials shown again verbatim (same `k`-image set). Each repeat **re-draws** its noisy distances
(a fresh, independent arrangement), so the original/repeat pair gives a within-subject
**test-retest reliability**:

$$\mathrm{rel}_s = \mathrm{mean}_{\text{repeated trials}} \; \rho\big(d^{\text{orig}},\, d^{\text{repeat}}\big)$$

the mean Spearman correlation between the original and repeat pairwise-distance vectors of the
subject's repeated trials (NaN for subjects with no repeats).

**Lever competition.** A repeat may only duplicate a *singles-only* trial (one with no
`frac_images_repeated`-doubled image), so no image exceeds 2 occurrences across both mechanisms.
At `images_per_trial = 20` even a modest `frac_images_repeated` saturates every trial with
doubled images and leaves none to repeat (`select_repeat_trials` raises). This notebook's grid
therefore fixes `frac_images_repeated = 0.0` and sweeps `frac_trials_repeated`, matching the
deployed `task_config.json`. The doubled-image **SNR** heuristic (which needs
`frac_images_repeated > 0`) is characterised in `evaluation_task_v2_3.ipynb` instead.

Everything downstream of trial generation - ground-truth generation, the MDS sweep, and the
coverage/stability metrics - is reused unchanged from `SpAM_Simulations/pipeline.py`.


In [ ]:
from pathlib import Path
from itertools import combinations, product

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express.colors as px_colors
import plotly.io as pio

from SpAM_Simulations.config import TaskV2_4SimulationConfig, MDSSweepConfig
from SpAM_Simulations import pipeline

pio.renderers.default = 'browser'

## Configure & Generate Simulation
Pick a `TaskV2_3SimulationConfig` - same levers as `SimulationConfig`
(`n_images`/`n_dims`, `num_subjects`, `trials_per_subject`, `images_per_trial`,
`subjects_noise_scale`, `subjects_noise_df`, `reps`, `seed`) plus the new
`frac_images_repeated`. The bundled small config runs end-to-end quickly; uncomment the
full-study config (matching `SpAM_Task/task_config.json`'s real `t=10, k=20` and the
725-image dataset) for the real run.


In [ ]:
# --- Study configuration ----------------------------------------------------------------
# Small configuration for a quick end-to-end run (uncomment to validate the notebook):
sim_config = TaskV2_4SimulationConfig(
    n_images=300,
    n_dims=6,
    num_subjects=[20, 40, 75],
    trials_per_subject=[12],
    images_per_trial=[20],
    subjects_noise_scale=[0.3, 0.6],
    subjects_noise_df=[3],
    frac_images_repeated=[0.0],
    frac_trials_repeated=[0.0, 0.1, 0.25],
    reps=3,
    seed=42,
)

# Full study configuration (uncomment for the real run - this is much heavier)
# mirrors the deployed task: frac_images_repeated fixed at 0.0, sweep frac_trials_repeated.
# sim_config = TaskV2_4SimulationConfig(
#     n_images=725,
#     n_dims=10,
#     num_subjects=[30, 50, 75, 250],
#     trials_per_subject=[10, 15, 20],
#     images_per_trial=[20],
#     subjects_noise_scale=[0.5, 0.8],
#     subjects_noise_df=[1],
#     frac_images_repeated=[0.0],
#     frac_trials_repeated=[0.0, 0.1, 0.2, 0.3],
#     reps=5,
#     seed=42,
# )

sim = pipeline.generate_task_v2_4_simulation(sim_config, verbose=True)


## Evaluate Simulation
### Coverage
Same three coverage scores as the original notebook (image coverage, pair coverage,
`P[connected]`), now also broken out by `frac_images_repeated`: a higher fraction of
repeated images means fewer *distinct* images per subject (`n_unique` shrinks), which can
lower coverage at a fixed subject count.


In [ ]:
coverage = pipeline.compute_coverage_table(sim).rename(columns={"num_subjects": "n_subjects"})
coverage["is_connected"] = coverage["num_connected_components"] == 1.0

# Coverage scores aren't affected by subject noise, so we average across those parameters
# (and across trials_per_subject/images_per_trial here, since the bundled config fixes them -
# extend the groupby below if you sweep those too).
coverage_summary = (
    coverage.groupby(["n_subjects", "frac_trials_repeated"])
    .agg(
        num_reps=("img_coverage", "size"),
        percent_img_coverage_mean=("img_coverage", "mean"),
        percent_img_coverage_sem=("img_coverage", "sem"),
        percent_pair_coverage_mean=("pair_coverage", "mean"),
        percent_pair_coverage_sem=("pair_coverage", "sem"),
        p_is_connected_mean=("is_connected", "mean"),
        p_is_connected_sem=("is_connected", "sem"),
    )
    .sort_index()
    .reset_index()
)

In [ ]:
ROW_TITLES = {
    "% IMG COVERAGE": "percent_img_coverage",
    "% PAIR COVERAGE": "percent_pair_coverage",
    "P[CONNECTED]": "p_is_connected",
}
frac_values = sorted(coverage_summary["frac_trials_repeated"].unique())
COL_TITLES = {f: f"frac_trials_repeated = {f:.3g}" for f in frac_values}
coverage_fig = make_subplots(
    rows=len(ROW_TITLES), cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="Number of Subjects",
    vertical_spacing=0.05, horizontal_spacing=0.025,
)
for c, frac in enumerate(frac_values):
    df = coverage_summary[coverage_summary["frac_trials_repeated"] == frac]
    for r, (row_title, prefix) in enumerate(ROW_TITLES.items()):
        coverage_fig.add_trace(
            row=r + 1, col=c + 1, trace=go.Scatter(
                x=df["n_subjects"], y=df[f"{prefix}_mean"],
                error_y=dict(type="data", array=df[f"{prefix}_sem"], visible=True),
                mode="lines+markers", line=dict(color=px_colors.qualitative.Plotly[r]),
                showlegend=False,
            )
        )
        if c == 0:
            coverage_fig.update_yaxes(
                row=r + 1, col=c + 1,
                title=dict(text=row_title, font=dict(size=14, color='black'))
            )
del c, r, frac, df, row_title, prefix

coverage_fig.update_layout(
    height=650, width=1500,
    title=dict(
        text="Coverage by Number of Subjects and Trial-Repetition Fraction",
        font=dict(size=20, color='black')
    ),
)
coverage_fig.show()

### Stability
Spearman (rank) correlation between different repetitions of the same experimental
configuration, faceted by `frac_images_repeated` and colored by the noise parameters - same
idea as the original notebook.


In [ ]:
PARAM_FIELDS = [
    "num_subjects", "trials_per_subject", "images_per_trial",
    "subjects_noise_scale", "subjects_noise_df", "frac_images_repeated", "frac_trials_repeated",
]

correlations = (
    pipeline.compute_stability_table(sim)
    .dropna(subset=["spearman"])
    .groupby(PARAM_FIELDS)
    .agg(count=("spearman", "size"), r_mean=("spearman", "mean"), r_sem=("spearman", "sem"))
)
correlations.index = correlations.index.set_names(
    ["n_subjects", "trials_per_subject", "images_per_trial",
     "subjects_noise_scale", "subjects_noise_df", "frac_images_repeated", "frac_trials_repeated"]
)


In [ ]:
frac_values = sorted(correlations.index.get_level_values("frac_trials_repeated").unique())
COL_TITLES = {f: f"frac_trials_repeated = {f:.3g}" for f in frac_values}
corr_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="Number of Subjects",
)
for c, frac in enumerate(frac_values):
    subset = correlations.loc[correlations.index.get_level_values("frac_trials_repeated") == frac]
    noise_scales = sorted(subset.index.get_level_values("subjects_noise_scale").unique())
    noise_dfs = sorted(subset.index.get_level_values("subjects_noise_df").unique())
    for i, (noise_scale, noise_df) in enumerate(product(noise_scales, noise_dfs)):
        name = f"Scale={noise_scale}<br>DFs={noise_df}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = subset.loc[
            (subset.index.get_level_values("subjects_noise_scale") == noise_scale) &
            (subset.index.get_level_values("subjects_noise_df") == noise_df)
        ]
        corr_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df.index.get_level_values("n_subjects"), y=df["r_mean"],
                error_y=dict(type="data", array=df["r_sem"], visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
            )
        )
del c, frac, subset, noise_scales, noise_dfs, i, noise_scale, noise_df, name, color, df

corr_fig.update_yaxes(row=1, col=1, title=dict(text="Spearman R", font=dict(size=14, color='black')))
corr_fig.update_layout(
    height=450, width=1500,
    title=dict(text="Pre-MDS Stability by Trial-Repetition Fraction", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subject Noise Parameters")),
)
corr_fig.show()

## Subject Test-Retest Reliability
Each subject who receives whole-trial repeats (`frac_trials_repeated > 0`) yields a test-retest
reliability: the mean Spearman correlation between the original and repeat presentations of
their repeated trials - exactly the kind of internal-consistency signal we could also compute
from real (ground-truth-free) data. It is NaN for subjects with no repeated trials (so it is
undefined for the entire `frac_trials_repeated = 0` slice). Two checks below: (1) its
distribution across subjects, and (2) whether it degrades with the configured noise lever - if
it didn't track noise, it would be useless as a real-data QC proxy.


In [ ]:
# Per-subject test-retest reliability for one representative configuration: most repeated
# trials (so a defined reliability actually exists), most subjects, heaviest-tailed and noisiest.
focus_params = max(
    sim._results,
    key=lambda p: (p.frac_trials_repeated, p.num_subjects, -p.subjects_noise_df, p.subjects_noise_scale),
)
all_rel = np.concatenate([res.subject_test_retest for res in sim._results[focus_params]])
finite_rel = all_rel[np.isfinite(all_rel)]

rel_hist_fig = go.Figure(go.Histogram(x=finite_rel, nbinsx=30))
rel_hist_fig.update_layout(
    width=700, height=400,
    title=dict(text=f"Subject Test-Retest Reliability Distribution<br><sup>{focus_params}</sup>"),
    xaxis=dict(title=dict(text="mean Spearman r (original vs. repeat trial)")),
    yaxis=dict(title=dict(text="Count")),
    template="plotly_white",
)
rel_hist_fig.show()
print(f"{np.mean(np.isnan(all_rel)):.1%} of subjects had no repeated trials (reliability undefined)")


In [ ]:
# How the test-retest reliability tracks the configured subject-noise lever, per
# frac_trials_repeated (the frac_trials_repeated == 0 slice is all-NaN and drops out).
rel_vs_noise = (
    coverage[np.isfinite(coverage["mean_test_retest"])]
    .groupby(["subjects_noise_scale", "frac_trials_repeated"])
    .agg(mean_rel=("mean_test_retest", "mean"), sem_rel=("mean_test_retest", "sem"))
    .reset_index()
)

rel_fig = go.Figure()
for frac in sorted(rel_vs_noise["frac_trials_repeated"].unique()):
    df = rel_vs_noise[rel_vs_noise["frac_trials_repeated"] == frac]
    rel_fig.add_trace(go.Scatter(
        x=df["subjects_noise_scale"], y=df["mean_rel"],
        error_y=dict(type="data", array=df["sem_rel"].fillna(0), visible=True),
        name=f"frac_trials_repeated = {frac:.3g}",
        mode="lines+markers",
    ))
del frac, df
rel_fig.update_layout(
    width=700, height=400,
    title=dict(text="Mean Test-Retest Reliability vs. Configured Subject Noise Scale"),
    xaxis=dict(title=dict(text="subjects_noise_scale")),
    yaxis=dict(title=dict(text="mean(test-retest Spearman r)")),
    template="plotly_white",
    legend=dict(title=dict(text="Trial-Repetition Fraction")),
)
rel_fig.show()


## Run MDS
Same MDS sweep as the original notebook - it operates only on each configuration's mean
observed distances + weight mask, so it's entirely agnostic to how those distances were
generated (task-v0.1's random trials vs. the task-v2.3 per-subject design here).


In [ ]:
from SpAM_Simulations.storage import ResultStore
MDS_STORE_PATH = Path("run-task-v2.4") / "mds_store"

# uncomment to run a full MDS sweep
# sweep_config = MDSSweepConfig(
#     min_ndim=2,
#     max_iters=500,
#     convergence_tol=1e-6,
#     precalc_init=False,
# )
# store = pipeline.run_mds_sweep(sim, sweep_config, MDS_STORE_PATH, parallel=False, verbose=True)

# uncomment to load existing MDS results
store = ResultStore.open(MDS_STORE_PATH)

mds_meta = store.metadata()
print(f"{len(mds_meta)} MDS results stored at {MDS_STORE_PATH}")

In [ ]:
from collections import Counter

Counter(mds_meta["status"])

### Scree Plot
MDS Stress vs target dimensionality, faceted by `frac_images_repeated`.


In [ ]:
success_mds_results = mds_meta[mds_meta["status"].isin(["success", "max_iters"])].copy()

frac_values = sorted(success_mds_results["frac_trials_repeated"].unique())
COL_TITLES = {f: f"frac_trials_repeated = {f:.3g}" for f in frac_values}
stress_scree_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="DIMENSIONS",
)
for c, frac in enumerate(frac_values):
    subset = success_mds_results[success_mds_results["frac_trials_repeated"] == frac]
    noise_scales = sorted(subset["subjects_noise_scale"].unique())
    n_subjs = sorted(subset["num_subjects"].unique())
    for i, (noise_scale, n_subj) in enumerate(product(noise_scales, n_subjs)):
        name = f"Scale={noise_scale}<br>N={n_subj}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = (
            subset[(subset["subjects_noise_scale"] == noise_scale) & (subset["num_subjects"] == n_subj)]
            .groupby("ndim")["stress"]
            .agg(N="count", mean="mean", sem="sem")
        )
        stress_scree_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df.index.get_level_values("ndim"), y=df["mean"],
                error_y=dict(type="data", array=df["sem"].fillna(0), visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
                hovertemplate=f"{name}<br>" + "NDIM=%{x}<br>Stress=%{y:.4f}",
            )
        )
del c, frac, subset, noise_scales, n_subjs, i, noise_scale, n_subj, name, color, df

stress_scree_fig.update_yaxes(row=1, col=1, title=dict(text="Stress", font=dict(size=14, color='black')))
stress_scree_fig.update_layout(
    height=500, width=1500,
    title=dict(text="MDS Stress (lower is better)", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subjects / Noise")),
)
stress_scree_fig.show()

### Embedding Stability
Post-MDS Spearman agreement of reconstructed distances (`confdist`) across repetitions,
faceted by `frac_images_repeated`.


In [ ]:
embedding_corrs_df = (
    pipeline.compute_embedding_stability(store)
    .rename(columns={"n_reps": "N", "mean_spearman": "r_mean", "sem_spearman": "r_sem"})
)

frac_values = sorted(embedding_corrs_df["frac_trials_repeated"].unique())
COL_TITLES = {f: f"frac_trials_repeated = {f:.3g}" for f in frac_values}
embedded_corr_fig = make_subplots(
    rows=1, cols=len(COL_TITLES),
    column_titles=list(COL_TITLES.values()),
    shared_xaxes=True, shared_yaxes=True,
    x_title="DIMENSIONS",
)
for c, frac in enumerate(frac_values):
    subset = embedding_corrs_df[embedding_corrs_df["frac_trials_repeated"] == frac]
    noise_scales = sorted(subset["subjects_noise_scale"].unique())
    n_subjs = sorted(subset["num_subjects"].unique())
    for i, (noise_scale, n_subj) in enumerate(product(noise_scales, n_subjs)):
        name = f"Scale={noise_scale}<br>N={n_subj}"
        color = px_colors.qualitative.Plotly[i % len(px_colors.qualitative.Plotly)]
        df = subset[(subset["subjects_noise_scale"] == noise_scale) & (subset["num_subjects"] == n_subj)]
        embedded_corr_fig.add_trace(
            row=1, col=c + 1, trace=go.Scatter(
                x=df["ndim"], y=df["r_mean"],
                error_y=dict(type="data", array=df["r_sem"].fillna(0), visible=True),
                name=name, legendgroup=name, showlegend=c == 0,
                mode="lines+markers", line=dict(color=color),
                hovertemplate=f"{name}<br>" + "NDIM=%{x}<br>Spearman's R=%{y:.4f}",
            )
        )
del c, frac, subset, noise_scales, n_subjs, i, noise_scale, n_subj, name, color, df

embedded_corr_fig.update_yaxes(row=1, col=1, title=dict(text="Spearman R", font=dict(size=14, color='black')))
embedded_corr_fig.update_layout(
    height=500, width=1500,
    title=dict(text="MDS-Stability (Spearman Correlation) by Trial-Repetition Fraction", font=dict(size=20, color='black')),
    legend=dict(title=dict(text="Subjects / Noise")),
)
embedded_corr_fig.show()

In [ ]:
embedded_corr_fig.show()